### Below is prep code for feature engineering

#### Objectives are to: 
 - assess data completeness 
 - impute and feature engineer columns with a lack of data
 - identify columns with multicollinearity

In [69]:
#load in modules
import os
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import scipy.stats as stats 
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
import time
from tabulate import tabulate
import re

In [68]:
#Load in dataset
df = pd.read_csv("../../../data/finaldatasets/testdata/RFgdpadded.csv", na_values=["n/a", "missing", "-", "", "NA", "N/A", "NA"])

#drop cols below 9<0
#df = df.loc[:, df.notna().mean() > 0.05]



#Observe values in rows if objects 
df["Soil_N"].dtype

df["Soil_N"].unique()
df["Soil_N"].apply(type).value_counts()

df['Soil_N'].loc[df['Soil_N'].apply(type) == str].unique()

/tmp/ipykernel_7487/2611002665.py:2: DtypeWarning: Columns (6,7,13,14,15,16,17,20,22,23,29,31,32,34,35,36,37,38,39,46,47,71,72,76,77,78) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../../../data/finaldatasets/testdata/RFgdpadded.csv", na_values=["n/a", "missing", "-", "", "NA", "N/A", "NA"])


array([], dtype=float64)

In [70]:
# data summary table
def summarize_column(col_data):
    numeric_data = pd.to_numeric(col_data, errors='coerce')
    is_numeric = numeric_data.notna().sum() > 0
    return {
        "dtype": col_data.dtype,
        "count": col_data.notna().sum(),
        "pct_complete": round(col_data.notna().mean() * 100, 1),
        "n_unique": col_data.nunique(dropna=True)
    }

# Build summary
summary_long = (
    pd.DataFrame([
        {"variable": col, **summarize_column(df[col])}
        for col in df.columns
    ])
    .sort_values(by="pct_complete", ascending=False)
    .reset_index(drop=True)
)

# Print in markdown table format
from tabulate import tabulate
print(tabulate(
    summary_long,
    headers="keys",
    tablefmt="github",
    showindex=False
))


| variable                       | dtype   |   count |   pct_complete |   n_unique |
|--------------------------------|---------|---------|----------------|------------|
| id                             | int64   |   78147 |          100   |      77753 |
| Observation.period             | int64   |   78147 |          100   |         42 |
| lon                            | float64 |   78147 |          100   |        251 |
| lat                            | float64 |   78147 |          100   |        251 |
| year                           | float64 |   78147 |          100   |         42 |
| Elevation                      | float64 |   78147 |          100   |        204 |
| pr_irrigated                   | float64 |   78143 |          100   |        198 |
| end_date                       | int64   |   78147 |          100   |         10 |
| Data.ID                        | object  |   78147 |          100   |         55 |
| Grain.yield..tons.ha.1.        | float64 |   78136 |          1

##### Imputation and feature engineering of the data


In [66]:

pd.set_option('display.max_rows', None)


#Observe values in rows if objects 

df["Pest.severity.score.cleaned"].dtype

list(df["Pest.severity.score.cleaned"].unique())

df["Pest.severity.score.cleaned"].value_counts(dropna=False).reset_index()
# df["irr_start_date"].apply(type).value_counts()

# df['irr_start_date'].loc[df['irr_start_date'].apply(type) == str].unique()

# treatmenttypes = list(df["Planting.date.1"].unique())

# treatmenttypes_df = pd.DataFrame(treatmenttypes, columns=["Planting.date.1"])

# treatmenttypes_df.to_csv("../../../data/finaldatasets/covariates/Planting.date.csv", index=False)

,Pest.severity.score.cleaned,count
0,NaN,68725
1,1.50,5289
2,5.00,661
3,0.00,595
4,9.00,410
5,10.00,381
6,1.00,231
7,22.00,229
8,39.50,207
9,15.00,190


Data Cleaning and resaving

In [ ]:
df["soil_type"] = pd.to_numeric(df["soil_type"], errors="coerce")  # converts to float, sets bad values to NaN
df.loc[df["soil_type"] < 1980, "soil_type"] = np.nan


#df = df.drop("irr_start_date_clean", axis=1)

##### save the dataset

In [51]:
#if you want to drop a col : 

df = df.drop("Month", axis=1)

In [71]:
df.to_csv("../../../data/finaldatasets/testdata/FixedRFdata.csv", index=False)

In [198]:
df = df.loc[:, df.notna().mean() > 0.05]

df.to_csv("../../../data/finaldatasets/testdata/FixedRFdata10%.csv", index=False)

In [135]:
df["Irrigation.mm_clean"] = pd.to_numeric(df["Irrigation..mm."], errors="coerce")

#create function that defines irrigation type
def classify_irrigation(val):
    if isinstance(val, str):
        val = val.lower()
        if "flooding" in val:
            return "flooding"
        elif "yes" in val:
            return "yes"
        elif "no" in val or "0.0" in val:
            return "no"
    try:
        return "mm_provided" if pd.notnull(pd.to_numeric(val)) else "NA"
    except:
        return "NA"
    
df["irrigation_applied"] = df["Irrigation..mm."].apply(classify_irrigation)


##### Seperate harvest date into julian days, weeks and months

In [ ]:
harvest_date = pd.to_datetime(df["Harvesting.date.1"], errors="coerce")

serial_fallback = pd.to_datetime(pd.to_numeric(df["Harvesting.date.1"], errors="coerce"), 
                                 origin="1899-12-30", unit="D", errors="coerce")

df["harvest_date_clean"] = harvest_date.fillna(serial_fallback)




In [162]:
df.loc[df["harvest_date_clean"].dt.year < 1980, "harvest_date_clean"] = pd.NaT

In [164]:
#getting days, months and years

df["harvest_dayofyear"] = df["harvest_date_clean"].dt.dayofyear
df["harvest_month"] = df["harvest_date_clean"].dt.month
df["harvest_week"] = df["harvest_date_clean"].dt.isocalendar().week


##### Create planting date, seperate into julian days, months weeks

In [46]:
def clean_date_value(val):
    """Clean individual date values"""
    if pd.isna(val):
        return np.nan
    
    val_str = str(val).strip()
    
    # Check if it's purely numeric
    if val_str.replace('.', '').replace('-', '').isdigit():
        # Remove decimal points for length check
        clean_num = val_str.replace('.0', '').replace('-', '')
        
        if len(clean_num) == 6:
            # Try to convert 6-digit numbers to dates
            try:
                # Try YYMMDD format first (like 160718 = 2016-07-18)
                year_part = int(clean_num[:2])
                month = int(clean_num[2:4])
                day = int(clean_num[4:6])
                
                # Convert 2-digit year to 4-digit
                if year_part > 50:  # Assume >50 means 19xx
                    year = 1900 + year_part
                else:  # <=50 means 20xx
                    year = 2000 + year_part
                
                # Validate the date
                test_date = datetime(year, month, day)
                return f"{month}/{day}/{year}"
                
            except:
                # If YYMMDD fails, try MMDDYY
                try:
                    month = int(clean_num[:2])
                    day = int(clean_num[2:4])
                    year_part = int(clean_num[4:6])
                    
                    if year_part > 50:
                        year = 1900 + year_part
                    else:
                        year = 2000 + year_part
                    
                    # Validate the date
                    test_date = datetime(year, month, day)
                    return f"{month}/{day}/{year}"
                except:
                    return np.nan  # Can't convert, make it NA
        else:
            # Single numbers or other lengths = error, convert to NA
            return np.nan
    
    # Keep non-numeric values (actual date strings)
    return val

def extract_date_features(date_str):
    """Extract julian day, month, and week from cleaned date"""
    if pd.isna(date_str):
        return pd.Series([np.nan, np.nan, np.nan])
    
    try:
        # Parse the cleaned date string
        date_obj = pd.to_datetime(date_str, format='%m/%d/%Y', errors='coerce')
        
        if pd.isna(date_obj):
            return pd.Series([np.nan, np.nan, np.nan])
        
        # Extract features
        julian_day = date_obj.timetuple().tm_yday  # Day of year (1-365/366)
        month = date_obj.month                     # Month (1-12)
        week = date_obj.isocalendar()[1]          # Week of year (1-52/53)
        
        return pd.Series([julian_day, month, week])
    
    except:
        return pd.Series([np.nan, np.nan, np.nan])

# Apply to your main dataset
print("Cleaning planting dates in main dataset...")

# Step 1: Clean the dates
print("Step 1: Cleaning date values...")
df["Planting.date.1"] = df["Planting.date.1"].apply(clean_date_value)

# Step 2: Extract date features
print("Step 2: Extracting date features...")
df[['Planting_Day', 'Planting_Month', 'Planting_Week']] = df["Planting.date.1"].apply(extract_date_features)

# Show results
print("\nCleaning complete!")
print(f"Total rows: {len(df)}")
print(f"Valid dates: {df['Julian_Day'].notna().sum()}")
print(f"NA values: {df['Julian_Day'].isna().sum()}")

print("\nSample cleaned data:")
print(df[['Planting.date.1', 'Julian_Day', 'Month', 'Week']].head(10))

print("\nMonth distribution:")
print(df['Month'].value_counts().sort_index())

print("\nJulian Day summary:")
print(df['Julian_Day'].describe())

Cleaning planting dates in main dataset...
Step 1: Cleaning date values...
Step 2: Extracting date features...

Cleaning complete!
Total rows: 78147
Valid dates: 8733
NA values: 69414

Sample cleaned data:
  Planting.date.1  Julian_Day  Month  Week
0             NaN         NaN    NaN   NaN
1             NaN         NaN    NaN   NaN
2             NaN         NaN    NaN   NaN
3             NaN         NaN    NaN   NaN
4             NaN         NaN    NaN   NaN
5             NaN         NaN    NaN   NaN
6             NaN         NaN    NaN   NaN
7             NaN         NaN    NaN   NaN
8             NaN         NaN    NaN   NaN
9             NaN         NaN    NaN   NaN

Month distribution:
Month
1.0      132
2.0      150
3.0      291
4.0       40
5.0       22
6.0       24
7.0     1355
8.0      332
9.0      610
10.0    4772
11.0     980
12.0      25
Name: count, dtype: int64

Julian Day summary:
count    8733.000000
mean      255.574946
std        65.363552
min        16.000000
25%    

Fix emmissions, create yes no flag

In [195]:

#fixing emmisions because values are so low: 
def classify_emissions(val):
    if pd.isna(val):
        return "NA"  # preserve NaN as "NA"

    if isinstance(val, str):
        val = val.lower()
        if val == "yes":
            return "yes"
        elif val == "no":
            return "no"

    try:
        num = float(val)
        return "yes" if num > 0 else "no"
    except:
        return "NA"
    
    
df["emissions_flag"] = df["Emissions..yes.no."].apply(classify_emissions)

##### define treatment categories

In [ ]:
#Define treatment categories
def classify_treatment(val):
    if pd.isna(val):
        return "NA"  # preserve NaN as "NA"
    
    if isinstance(val, str):
        t = val.lower().strip()
        
        # 1. Control treatments
        if t in ['control', 'ck', '0', 'check'] or 'control' in t or 'no fertil' in t or 'unfertil' in t:
            return "Control"
        
        # 2. Enhanced Efficiency Fertilizers (check first)
        if ('inhibitor' in t or 'slow release' in t or 'controlled release' in t or
            'coated' in t or 'dcd' in t or 'nitrapyrin' in t or 'nbpt' in t or
            'dmpp' in t or 'polymer' in t or 'crf' in t or 'srf' in t or 'enhanced' in t or 'stabilized' in t):
            return "Enhanced Efficiency"
        
        # 3. Combined Organic-Inorganic (check before pure categories)
        has_organic = any(word in t for word in ['manure', 'straw', 'organic', 'biochar', 'compost', 'residue'])
        has_inorganic = any(word in t for word in ['urea', 'npk', 'fertilizer', 'chemical', 'mineral'])
        if has_organic and has_inorganic:
            return "Combined Organic-Inorganic"
        
        # 4. Inorganic Nitrogen Fertilizers
        if (('urea' in t or 'ammonium' in t or 'nitrate' in t or 'anhydrous' in t or 
             'sulfate' in t or 'liquid ammonia' in t) and not has_organic):
            return "Inorganic Nitrogen"
        
        # 5. NPK Complex Fertilizers
        if ('npk' in t or 'dap' in t or 'diammonium phosphate' in t or 'compound fertilizer' in t):
            return "NPK Complex"
        
        # 6. Organic Animal-based
        if ('manure' in t or 'dung' in t or 'slurry' in t or 'pig' in t or 'cattle' in t or 
            'chicken' in t or 'poultry' in t or 'fym' in t or 'compost' in t):
            return "Organic Animal-based"
        
        # 7. Crop Residue & Plant-based
        if ('straw' in t or 'residue' in t or 'biochar' in t or 'mulch' in t or 'biomass' in t):
            return "Crop Residue & Plant-based"
        
        # 8. Mineral Fertilizers
        if ('mineral fertilizer' in t or 'chemical fertilizer' in t or 'synthetic' in t or
            ('fertilizer' in t and 'organic' not in t)):
            return "Mineral Fertilizers"
        
        # 9. Nitrogen Rate Treatments
        if ('nitrogen' in t or any(char in t for char in ['n0', 'n1', 'n2', 'n3', 'n4', 'n5']) or 
            'kg.*n' in t or 'n rate' in t):
            return "Nitrogen Rate Treatments"
        
        # 10. Phosphorus & Potassium
        if (('phosph' in t or 'potash' in t or 'potassium' in t or 'pk' in t) and 'npk' not in t):
            return "Phosphorus & Potassium"
        
        # 11. Micronutrients & Secondary
        if ('zinc' in t or 'boron' in t or 'iron' in t or 'manganese' in t or 'sulfur' in t or 
            'foliar' in t or 'micro' in t):
            return "Micronutrients & Secondary"
        
        # 12. Application Methods
        if ('broadcast' in t or 'band' in t or 'drip' in t or 'split' in t or 'placement' in t):
            return "Application Methods"
        
        # 13. Specialty Treatments
        if ('pesticide' in t or 'herbicide' in t or 'fungicide' in t or 'stress' in t or 
            'drought' in t or 'tillage' in t):
            return "Specialty Treatments"
    
    return "Other"

# Create the new classified column
df["Treatment_Category"] = df["Treatment.type"].apply(classify_treatment)

# Optional: Check the results
print("Classification Results:")
print(df["Treatment_Category"].value_counts())

# Optional: See examples side by side
print("\nExample classifications:")
print(df[["Treatment.type", "Treatment_Category"]].head(10))

Classification Results:
Treatment_Category
NA                            54722
Other                          7085
Nitrogen Rate Treatments       7061
Specialty Treatments           4355
Micronutrients & Secondary     2130
Organic Animal-based            932
Inorganic Nitrogen              435
NPK Complex                     384
Phosphorus & Potassium          305
Control                         261
Combined Organic-Inorganic      235
Enhanced Efficiency             130
Mineral Fertilizers              66
Crop Residue & Plant-based       37
Application Methods               9
Name: count, dtype: int64

Example classifications:
                      Treatment.type  Treatment_Category
0                            Control             Control
1                         urea 0+91U  Inorganic Nitrogen
2     ammonium sulfate+urea 60AS+91U  Inorganic Nitrogen
3  anhydrous ammonium+urea 112AA+91U  Inorganic Nitrogen
4    ammonium sulfate+urea 163AS+91U  Inorganic Nitrogen
5                      

#### Define N.type categories

In [ ]:
#Classify nitrogen into categories (might need work)

def classify_nitrogen_type(val):
    """
    Classify nitrogen fertilizer types based on chemical form, technology, and amendments.
    More specialized than general fertilizer classification.
    
    Categories based on nitrogen fertilizer science literature:
    1. Chemical form (urea, ammonium, nitrate, anhydrous)
    2. Technology level (conventional vs enhanced efficiency)
    3. Organic amendments integration
    """
    
    if pd.isna(val):
        return "NA"  # preserve NaN as "NA"
    
    if isinstance(val, str):
        t = val.lower().strip()
        
        # 1. No Nitrogen / Control
        if (t in ['control', 'ck', '0', 'check'] or 'control' in t or 
            'no n' in t or 'unfertil' in t or 'no fertil' in t):
            return "No Nitrogen"
        
        # 2. Enhanced Efficiency Urea (check before pure urea)
        if ('urea' in t and 
            ('inhibitor' in t or 'dcd' in t or 'nbpt' in t or 'nitrapyrin' in t or 
             'coated' in t or 'controlled' in t or 'slow release' in t or 
             'enhanced' in t or 'stabilized' in t or 'polymer' in t)):
            return "Enhanced Efficiency Urea"
        
        # 3. Urea with Organic Amendments (check before pure urea)
        if ('urea' in t and 
            ('straw' in t or 'manure' in t or 'biochar' in t or 'organic' in t or 
             'compost' in t or 'fym' in t or 'residue' in t) and
            not ('inhibitor' in t or 'dcd' in t or 'nbpt' in t or 'coated' in t)):
            return "Urea + Organic"
        
        # 4. Pure Urea (conventional urea without additives)
        if ('urea' in t and 
            not ('+' in t or 'inhibitor' in t or 'dcd' in t or 'nbpt' in t or 
                 'coated' in t or 'straw' in t or 'manure' in t or 'biochar' in t or
                 'controlled' in t or 'slow' in t)):
            return "Pure Urea"
        
        # 5. Mixed Nitrogen Sources (combinations of different N forms)
        if ('+' in t and 
            ((('urea' in t and 'ammonium' in t) or 
              ('urea' in t and 'nitrate' in t) or 
              ('ammonium' in t and 'nitrate' in t)) and
             not ('straw' in t or 'manure' in t or 'biochar' in t))):
            return "Mixed N Sources"
        
        # 6. Ammonium-based (not mixed with urea)
        if (('ammonium' in t or 'anhydrous' in t) and 'urea' not in t):
            # Check if enhanced efficiency
            if ('inhibitor' in t or 'dcd' in t or 'dmpp' in t or 'coated' in t or 'controlled' in t):
                return "Enhanced Ammonium"
            else:
                return "Ammonium-based"
        
        # 7. Nitrate-based (not mixed with other N forms)
        if ('nitrate' in t and 'urea' not in t and 'ammonium' not in t):
            return "Nitrate-based"
        
        # 8. NPK Complex (nitrogen as part of complex fertilizer)
        if ('npk' in t or 'compound fertilizer' in t or 'complex fertilizer' in t):
            return "NPK Complex"
        
        # 9. Organic Nitrogen Sources
        if (('manure' in t or 'compost' in t or 'fym' in t or 'slurry' in t) and
            'urea' not in t and 'ammonium' not in t and 'nitrate' not in t):
            return "Organic N Sources"
        
        # 10. Liquid N Formulations
        if ('liquid' in t and ('nitrogen' in t or 'urea' in t or 'ammonium' in t)):
            return "Liquid N"
        
        # 11. Specialty N Formulations (foliar, custom blends, etc.)
        if ('foliar' in t or 'custom' in t or 'blend' in t or 'specialty' in t):
            return "Specialty N"
    
    return "Other N"

# Create the new classified column for nitrogen types
df["N_Category"] = df["N.type"].apply(classify_nitrogen_type)

# Optional: Check the results
print("Nitrogen Classification Results:")
print(df["N_Category"].value_counts())

# Optional: See examples side by side
print("\nExample nitrogen classifications:")
print(df[["N.type", "N_Category"]].head(15))

Nitrogen Classification Results:
N_Category
NA                          62810
Other N                     13774
Pure Urea                     801
No Nitrogen                   156
Urea + Organic                147
NPK Complex                   147
Ammonium-based                122
Enhanced Efficiency Urea       90
Organic N Sources              73
Mixed N Sources                22
Nitrate-based                   3
Enhanced Ammonium               2
Name: count, dtype: int64

Example nitrogen classifications:
                               N.type                N_Category
0                                   0               No Nitrogen
1                          urea 0+91U                   Other N
2      ammonium sulfate+urea 60AS+91U           Mixed N Sources
3   anhydrous ammonium+urea 112AA+91U           Mixed N Sources
4     ammonium sulfate+urea 163AS+91U           Mixed N Sources
5                                   0               No Nitrogen
6      ammonium sulfate+urea 56AS+98U  

In [65]:
#Pest Severity Score
def clean_and_midpoint(value):
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip()
    
    # Replace strange Unicode dashes (like en-dash, em-dash, weird encodings) with hyphen
    value_str = re.sub(r"[^\d\-\.]", "", value_str.replace("â", "-").replace("–", "-").replace("—", "-"))

    # Match patterns like "15-29" or "70-84"
    if "-" in value_str:
        parts = value_str.split("-")
        try:
            parts = [float(p) for p in parts]
            return sum(parts) / 2
        except ValueError:
            return np.nan
    else:
        try:
            return float(value_str)
        except ValueError:
            return np.nan

# Apply to your column
df["Pest.severity.score.cleaned"] = df["Pest.severity.score.......66"].apply(clean_and_midpoint)

# Optional: inspect result
print(df[["Pest.severity.score.......66", "Pest.severity.score.cleaned"]].head(10))


  Pest.severity.score.......66  Pest.severity.score.cleaned
0                          NaN                          NaN
1                          NaN                          NaN
2                          NaN                          NaN
3                          NaN                          NaN
4                          NaN                          NaN
5                          NaN                          NaN
6                          NaN                          NaN
7                          NaN                          NaN
8                          NaN                          NaN
9                          NaN                          NaN
